In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


In [3]:
df = pd.read_csv("epl_final.csv")

print("Primeiras 5 Linhas:")
display(df.head())

print(f"\nDimensões:")
print(f"Linhas (partidas): {df.shape[0]}")
print(f"Colunas (variáveis): {df.shape[1]}")

print(f"\nColunas Disponíveis")
print(df.columns.tolist())

print(f"\nInformações Gerais:")
df.info()

Primeiras 5 Linhas:


,Season,MatchDate,HomeTeam,AwayTeam,FullTimeHomeGoals,FullTimeAwayGoals,FullTimeResult,HalfTimeHomeGoals,HalfTimeAwayGoals,HalfTimeResult,...,HomeShotsOnTarget,AwayShotsOnTarget,HomeCorners,AwayCorners,HomeFouls,AwayFouls,HomeYellowCards,AwayYellowCards,HomeRedCards,AwayRedCards
0,2000/01,2000-08-19,Charlton,Man City,4,0,H,2,0,H,...,14,4,6,6,13,12,1,2,0,0
1,2000/01,2000-08-19,Chelsea,West Ham,4,2,H,1,0,H,...,10,5,7,7,19,14,1,2,0,0
2,2000/01,2000-08-19,Coventry,Middlesbrough,1,3,A,1,1,D,...,3,9,8,4,15,21,5,3,1,0
3,2000/01,2000-08-19,Derby,Southampton,2,2,D,1,2,A,...,4,6,5,8,11,13,1,1,0,0
4,2000/01,2000-08-19,Leeds,Everton,2,0,H,2,0,H,...,8,6,6,4,21,20,1,3,0,0



Dimensões:
Linhas (partidas): 9380
Colunas (variáveis): 22

Colunas Disponíveis
['Season', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FullTimeHomeGoals', 'FullTimeAwayGoals', 'FullTimeResult', 'HalfTimeHomeGoals', 'HalfTimeAwayGoals', 'HalfTimeResult', 'HomeShots', 'AwayShots', 'HomeShotsOnTarget', 'AwayShotsOnTarget', 'HomeCorners', 'AwayCorners', 'HomeFouls', 'AwayFouls', 'HomeYellowCards', 'AwayYellowCards', 'HomeRedCards', 'AwayRedCards']

Informações Gerais:
<class 'pandas.DataFrame'>
RangeIndex: 9380 entries, 0 to 9379
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Season             9380 non-null   str  
 1   MatchDate          9380 non-null   str  
 2   HomeTeam           9380 non-null   str  
 3   AwayTeam           9380 non-null   str  
 4   FullTimeHomeGoals  9380 non-null   int64
 5   FullTimeAwayGoals  9380 non-null   int64
 6   FullTimeResult     9380 non-null   str  
 7   HalfTimeHomeGoals  93

In [4]:
print("Temporadas Disponíveis:")
print(sorted(df["Season"].unique()))

Temporadas Disponíveis:
['2000/01', '2001/02', '2002/03', '2003/04', '2004/05', '2005/06', '2006/07', '2007/08', '2008/09', '2009/10', '2010/11', '2011/12', '2012/13', '2013/14', '2014/15', '2015/16', '2016/17', '2017/18', '2018/19', '2019/20', '2020/21', '2021/22', '2022/23', '2023/24', '2024/25']


In [5]:
# Últimas 10 temporadas
temporadas = ['2015/16', '2016/17', '2017/18', '2018/19', '2019/20',
              '2020/21', '2021/22', '2022/23', '2023/24', '2024/25']

df = df[df["Season"].isin(temporadas)].copy()

print(f"Partidas pós filtro: {len(df)}")

# Saldo de Gols (Home - Away)
df["GoalDifference"] = df["FullTimeHomeGoals"] - df["FullTimeAwayGoals"]

# Total de gols 
df["TotalGoals"] = df["FullTimeHomeGoals"] + df["FullTimeAwayGoals"]

# Variável: 1 = vitória do time da casa / 0 = empate ou derrota
df["HomeWin"] = (df["FullTimeResult"] == "H").astype(int)

# Variável: 1 = vitória do time visitante
df["AwayWin"] = (df["FullTimeResult"] == "A").astype(int)

# Variável: 1 = empate
df["Draw"] = (df["FullTimeResult"] == "D").astype(int)

# Temporadas pandêmicas (sem torcida)
temporadas_pandemia = ["2019/20", "2020/21"]
df["Pandemic"] = df["Season"].isin(temporadas_pandemia).astype(int)

# Era: classifica cada temporada em 3 períodos
def classificar_era(season):
    if season in ["2015/16", "2016/17", "2017/18", "2018/19"]:
        return "Pré-pandemia"
    elif season in ["2019/20", "2020/21"]:
        return "Pandemia"
    else:
        return "Pós-pandemia"

df["Era"] = df["Season"].apply(classificar_era)

# Confirmando o resultado
print("\n=== NOVAS COLUNAS CRIADAS ===")
display(df[["Season", "HomeTeam", "AwayTeam", "FullTimeResult",
            "GoalDifference", "HomeWin", "Pandemic", "Era"]].head(10))

print("\n=== DISTRIBUIÇÃO POR ERA ===")
print(df["Era"].value_counts())

Partidas pós filtro: 3770

=== NOVAS COLUNAS CRIADAS ===


,Season,HomeTeam,AwayTeam,FullTimeResult,GoalDifference,HomeWin,Pandemic,Era
5610,2015/16,Bournemouth,Aston Villa,A,-1,0,0,Pré-pandemia
5611,2015/16,Chelsea,Swansea,D,0,0,0,Pré-pandemia
5612,2015/16,Everton,Watford,D,0,0,0,Pré-pandemia
5613,2015/16,Leicester,Sunderland,H,2,1,0,Pré-pandemia
5614,2015/16,Man United,Tottenham,H,1,1,0,Pré-pandemia
5615,2015/16,Norwich,Crystal Palace,A,-2,0,0,Pré-pandemia
5616,2015/16,Arsenal,West Ham,A,-2,0,0,Pré-pandemia
5617,2015/16,Newcastle,Southampton,D,0,0,0,Pré-pandemia
5618,2015/16,Stoke,Liverpool,A,-1,0,0,Pré-pandemia
5619,2015/16,West Brom,Man City,A,-3,0,0,Pré-pandemia



=== DISTRIBUIÇÃO POR ERA ===
Era
Pré-pandemia    1520
Pós-pandemia    1490
Pandemia         760
Name: count, dtype: int64


In [6]:
# Analisando Home Advantage
print("Análise Geral Do Home Advantage (2015-2025)\n")

# Métricas das temporadas em geral:
home_win_pct   = df["HomeWin"].mean() * 100
away_win_pct   = df["AwayWin"].mean() * 100
draw_pct       = df["Draw"].mean() * 100
avg_home_goals = df["FullTimeHomeGoals"].mean()
avg_away_goals = df["FullTimeAwayGoals"].mean()

print(f"Vitórias em casa:     {home_win_pct:.1f}%")
print(f"Vitórias fora:        {away_win_pct:.1f}%")
print(f"Empates:              {draw_pct:.1f}%")
print(f"Média de gols (casa): {avg_home_goals:.2f}")
print(f"Média de gols (fora): {avg_away_goals:.2f}")

#SMétricas separadas por era:
print("\n=== HOME ADVANTAGE POR ERA ===\n")

era_stats = df.groupby("Era").agg(
    Partidas        = ("HomeWin", "count"),
    Vitorias_Casa   = ("HomeWin", "mean"),
    Vitorias_Fora   = ("AwayWin", "mean"),
    Empates         = ("Draw", "mean"),
    Media_Gols_Casa = ("FullTimeHomeGoals", "mean"),
    Media_Gols_Fora = ("FullTimeAwayGoals", "mean"),
    Saldo_Medio     = ("GoalDifference", "mean")
).round(3)

# Conversão para porcentagem:
era_stats["Vitorias_Casa"] = (era_stats["Vitorias_Casa"] * 100).round(1)
era_stats["Vitorias_Fora"] = (era_stats["Vitorias_Fora"] * 100).round(1)
era_stats["Empates"]       = (era_stats["Empates"] * 100).round(1)

# Ordenação por período:
ordem = ["Pré-pandemia", "Pandemia", "Pós-pandemia"]
era_stats = era_stats.reindex(ordem)

display(era_stats)

Análise Geral Do Home Advantage (2015-2025)

Vitórias em casa:     44.6%
Vitórias fora:        32.1%
Empates:              23.4%
Média de gols (casa): 1.55
Média de gols (fora): 1.28

=== HOME ADVANTAGE POR ERA ===



,Partidas,Vitorias_Casa,Vitorias_Fora,Empates,Media_Gols_Casa,Media_Gols_Fora,Saldo_Medio
Era,,,,,,,
Pré-pandemia,1520,45.9,30.3,23.8,1.547,1.203,0.345
Pandemia,760,41.6,35.4,23.0,1.434,1.274,0.161
Pós-pandemia,1490,44.7,32.1,23.2,1.623,1.356,0.266


In [ ]:
# Dados das partidas por era:
eras = ["Pré-pandemia", "Pandemia", "Pós-pandemia"]
vit_casa = [45.9, 41.6, 44.7]
vit_fora = [30.3, 35.4, 32.1]
empates  = [23.8, 23.0, 23.2]

fig = go.Figure()

fig.add_trace(go.Bar(
    name="Vitória Casa",
    x=eras,
    y=vit_casa,
    marker_color="#1a78cf",
    text=[f"{v}%" for v in vit_casa],
    textposition="outside"
))

fig.add_trace(go.Bar(
    name="Vitória Fora",
    x=eras,
    y=vit_fora,
    marker_color="#e8473f",
    text=[f"{v}%" for v in vit_fora],
    textposition="outside"
))

fig.add_trace(go.Bar(
    name="Empate",
    x=eras,
    y=empates,
    marker_color="#f0a500",
    text=[f"{v}%" for v in empates],
    textposition="outside"
))

fig.update_layout(
    title="Distribuição De Resultados Por Período — Premier League (2015-2025)",
    xaxis_title="Era",
    yaxis_title="Percentual De Partidas (%)",
    barmode="GDroup",
    yaxis=dict(range=[0, 55]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0.3),
    template="plotly_white",
    height=500
)

fig.show()

In [10]:
# Vitórias em casa por temporada
season_stats = df.groupby("Season").agg(
    Vitorias_Casa=("HomeWin", "mean"),
    Vitorias_Fora=("AwayWin", "mean")
).reset_index()

season_stats["Vitorias_Casa"] = (season_stats["Vitorias_Casa"] * 100).round(1)
season_stats["Vitorias_Fora"] = (season_stats["Vitorias_Fora"] * 100).round(1)

fig = go.Figure()

# Linha de vitórias em casa
fig.add_trace(go.Scatter(
    x=season_stats["Season"],
    y=season_stats["Vitorias_Casa"],
    mode="lines+markers",
    name="Vitória Casa",
    line=dict(color="#1a78cf", width=2.5),
    marker=dict(size=8)
))

# Linha de vitórias fora
fig.add_trace(go.Scatter(
    x=season_stats["Season"],
    y=season_stats["Vitorias_Fora"],
    mode="lines+markers",
    name="Vitória Fora",
    line=dict(color="#e8473f", width=2.5),
    marker=dict(size=8)
))

# Faixa destacando as temporadas pandêmicas
fig.add_vrect(
    x0="2019/20", x1="2020/21",
    fillcolor="orange", opacity=0.15,
    layer="below", line_width=0,
    annotation_text="  Pandêmia",
    annotation_position="top left",
    annotation_font_size=12
)

fig.update_layout(
    title="Evolução do home advantage por temporada — Premier League (2015-2025)",
    xaxis_title="Temporada",
    yaxis_title="Percentual de vitórias (%)",
    yaxis=dict(range=[20, 60]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0.35),
    template="plotly_white",
    height=500
)

fig.show()

In [12]:
# Aproveitamento em casa por time
home_stats = df.groupby("HomeTeam").agg(
    Jogos_Casa=("HomeWin", "count"),
    Vitorias_Casa=("HomeWin", "mean")
).reset_index()
home_stats.columns = ["Team", "Jogos_Casa", "Vitorias_Casa"]

# Aproveitamento fora por time
away_stats = df.groupby("AwayTeam").agg(
    Jogos_Fora=("AwayWin", "count"),
    Vitorias_Fora=("AwayWin", "mean")
).reset_index()
away_stats.columns = ["Team", "Jogos_Fora", "Vitorias_Fora"]

# Juntando os dois (só times que aparecem nos dois lados)
team_stats = home_stats.merge(away_stats, on="Team", how="inner")

# Convertendo para percentual
team_stats["Vitorias_Casa"] = (team_stats["Vitorias_Casa"] * 100).round(1)
team_stats["Vitorias_Fora"] = (team_stats["Vitorias_Fora"] * 100).round(1)

# Diferença entre casa e fora
team_stats["Diferenca"] = (team_stats["Vitorias_Casa"] - team_stats["Vitorias_Fora"]).round(1)

# Filtrando times com mínimo de 100 jogos em casa
team_stats = team_stats[team_stats["Jogos_Casa"] >= 100]

# Ordenando pela diferença
team_stats = team_stats.sort_values("Diferenca", ascending=True)

# Gráfico
fig = go.Figure()

fig.add_trace(go.Bar(
    y=team_stats["Team"],
    x=team_stats["Diferenca"],
    orientation="h",
    marker_color=[
        "#e8473f" if v < 0 else "#1a78cf"
        for v in team_stats["Diferenca"]
    ],
    text=[f"{v}pp" for v in team_stats["Diferenca"]],
    textposition="outside"
))

fig.add_vline(x=0, line_width=1.5, line_color="gray")

fig.update_layout(
    title="Dependência do Fator Home/Away Por Time — Premier League (2015-2025)<br><sup>Diferença entre % vitórias em casa e % vitórias fora</sup>",
    xaxis_title="Diferença (Pontos Percentuais)",
    yaxis_title="",
    template="plotly_white",
    height=700,
    margin=dict(l=150)
)

fig.show()

In [13]:
# Média de gols por temporada
goals_season = df.groupby("Season").agg(
    Media_Casa=("FullTimeHomeGoals", "mean"),
    Media_Fora=("FullTimeAwayGoals", "mean")
).reset_index().round(2)

fig = go.Figure()

# Área preenchida de fora
fig.add_trace(go.Scatter(
    x=goals_season["Season"],
    y=goals_season["Media_Fora"],
    fill="tozeroy",
    mode="lines",
    name="Gols fora (média)",
    line=dict(color="#e8473f", width=2),
    fillcolor="rgba(232, 71, 63, 0.15)"
))

# Área preenchida de casa 
fig.add_trace(go.Scatter(
    x=goals_season["Season"],
    y=goals_season["Media_Casa"],
    fill="tonexty",
    mode="lines",
    name="Gols casa (média)",
    line=dict(color="#1a78cf", width=2),
    fillcolor="rgba(26, 120, 207, 0.2)"
))

# Faixa pandêmica
fig.add_vrect(
    x0="2019/20", x1="2020/21",
    fillcolor="orange", opacity=0.12,
    layer="below", line_width=0,
    annotation_text="Sem torcida",
    annotation_position="top left",
    annotation_font_size=11
)

fig.update_layout(
    title="Média de gols por partida — Casa vs Fora<br><sup>Premier League (2015-2025)</sup>",
    xaxis_title="Temporada",
    yaxis_title="Média de gols por partida",
    yaxis=dict(range=[0.8, 2.2]),
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0.3),
    height=480
)

fig.show()

In [15]:
# Médias de cartões e faltas por home vs away e era
disciplina = df.groupby("Era").agg(
    Faltas_Casa      =("HomeFouls", "mean"),
    Faltas_Fora      =("AwayFouls", "mean"),
    Amarelos_Casa    =("HomeYellowCards", "mean"),
    Amarelos_Fora    =("AwayYellowCards", "mean"),
    Vermelhos_Casa   =("HomeRedCards", "mean"),
    Vermelhos_Fora   =("AwayRedCards", "mean"),
    Chutes_Casa      =("HomeShots", "mean"),
    Chutes_Fora      =("AwayShots", "mean"),
    Chutes_Alvo_Casa =("HomeShotsOnTarget", "mean"),
    Chutes_Alvo_Fora =("AwayShotsOnTarget", "mean"),
).round(2)

# Reordenando cronologicamente
disciplina = disciplina.reindex(["Pré-pandemia", "Pandemia", "Pós-pandemia"])

# Transpondo para o heatmap ficar com variáveis nas linhas (linha para coluna)
disciplina_T = disciplina.T

fig = px.imshow(
    disciplina_T,
    text_auto=True,
    color_continuous_scale="Blues",
    aspect="auto",
    title="Métricas de Comportamento em Campo por Era — Casa vs Fora<br><sup>Premier League (2015-2025)</sup>"
)

fig.update_layout(
    xaxis_title="Era",
    yaxis_title="Métrica",
    height=480,
    coloraxis_showscale=False
)

fig.show()

# Tabela resumo de diferenças
print("\n=== DIFERENÇA CASA vs FORA (média geral) ===\n")
print(f"Faltas:           Casa {df['HomeFouls'].mean():.2f}  |  Fora {df['AwayFouls'].mean():.2f}")
print(f"Cartões amarelos: Casa {df['HomeYellowCards'].mean():.2f}  |  Fora {df['AwayYellowCards'].mean():.2f}")
print(f"Chutes:           Casa {df['HomeShots'].mean():.2f}  |  Fora {df['AwayShots'].mean():.2f}")
print(f"Chutes ao alvo:   Casa {df['HomeShotsOnTarget'].mean():.2f}  |  Fora {df['AwayShots'].mean():.2f}")


=== DIFERENÇA CASA vs FORA (média geral) ===

Faltas:           Casa 10.54  |  Fora 10.92
Cartões amarelos: Casa 1.63  |  Fora 1.81
Chutes:           Casa 13.92  |  Fora 11.49
Chutes ao alvo:   Casa 4.78  |  Fora 11.49


## 📊 Conclusões da Análise

### O Home Advantage é real e mensurável!
Na Premier League entre 2015 e 2025, times da casa venceram **44.6%** das partidas contra 
**32.1%** dos visitantes, isso representou uma diferença de 12.5 pontos percentuais 
que confirma que jogar em casa é uma vantagem concreta, não apenas percepção da torcida 
ou apaixonados por futebol.

### Os dados comprovaram que a torcida faz a DIFERENÇA!
O experimento natural da pandemia revelou o impacto direto da torcida:
- Vitórias em casa caíram de **45.9%** para **41.6%** sem público;
- Vitórias fora de casa subiram de **30.3%** para **35.4%**;
- O saldo médio de gols por jogo caiu de **+0.345** para **+0.161**.

Retirar a torcida praticamente reduziu o Home Advantage à metade.

### A vantagem não voltou ao nível original.
Após a pandemia, o Home Advantage se recuperou parcialmente (44.7%), mas os
visitantes continuam marcando mais gols do que no período pré-pandemia. Os times
aprenderam a jogar sem pressão de torcida e mantiveram esse comportamento
mais ofensivo fora de casa.

### Nem todos os times dependem igual do fator campo.
Tottenham (21.5pp) e Arsenal (21.4pp) são os times que mais dependem de casa,
enquanto Crystal Palace (4.6pp) e Southampton (4.9pp) performam de forma
semelhante em qualquer lugar. Times com torcidas mais intensas e estádios
modernos amplificam o efeito.

### O visitante comete mais faltas e leva mais cartões.
Em média, o time visitante comete mais faltas (10.92 vs 10.54) e recebe mais
cartões amarelos (1.81 vs 1.63), indicando que a pressão fora de casa
influencia o comportamento em campo além dos resultados.

---
*Análise realizada com dados de 3.770 partidas da Premier League (2015-2026).*
*Dataset: English Premier League Match Data 2000-2025 (Kaggle)*